In [ ]:
import pandas as pd
from neo4j import GraphDatabase

# Step 1: Connect to the Neo4j database
# Replace with your own Neo4j connection details
uri = "bolt://localhost:7687"
username = "neo4j"
password = "neo4jsumukhi"

# Read the CSV file
df = pd.read_csv('argo_labeled.csv')

# Step 2: Write data to Neo4j
def create_graph_from_csv(tx, row_data):
    """
    This function takes a single row of the DataFrame and creates the corresponding
    nodes and relationships in the Neo4j database using MERGE.
    MERGE is used to ensure that nodes and relationships are not created if they already exist.
    """
    # Create or match the Location node
    tx.run("""
        MERGE (loc:Location {
            latitude: $latitude,
            longitude: $longitude
        })
    """, latitude=row_data['latitude'], longitude=row_data['longitude'])
    
    # Create or match the Measurement node
    tx.run("""
        CREATE (m:Measurement {
            depth: $depth,
            temperature: $temperature,
            oxygen: $oxygen,
            chlorophyll: $chlorophyll,
            salinity: $salinity,
            anomaly: $anomaly
        })
    """, **row_data)
    
    # Create the relationships
    tx.run("""
        MATCH (loc:Location {latitude: $latitude, longitude: $longitude})
        MATCH (m:Measurement {depth: $depth})
        MERGE (m)-[:MEASURED_AT]->(loc)
    """, **row_data)

# Main execution block
with GraphDatabase.driver(uri, auth=(username, password)) as driver:
    with driver.session() as session:
        # Loop through the DataFrame and create the graph
        for index, row in df.iterrows():
            session.write_transaction(create_graph_from_csv, row.to_dict())

print("Data successfully loaded into Neo4j.")

C:\Users\priya\AppData\Local\Temp\ipykernel_39172\2109305175.py:52: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_graph_from_csv, row.to_dict())


AuthError: {code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.}